### Analyze Visium H&E data

**Visium spots based purely on the tissue image**

This tutorial shows how to apply Squidpy for the analysis of Visium spatial transcriptomics data.

https://squidpy.readthedocs.io/en/stable/notebooks/tutorials/tutorial_visium_hne.html

#### Data

dataset portal <https://support.10xgenomics.com/spatial-gene-expression/datasets>



In [ ]:
import numpy as np
import pandas as pd
import os, sys

import anndata as ad
import scanpy as sc
import squidpy as sq
import liana as li
from pathlib import Path

ROOT0 = Path("/home/flavio/uv/gwas/")
ROOT_SRC = ROOT0 / "src"

sys.path.append(str(ROOT_SRC))

from libs.Basic import create_dir, pdwritecsv, pdreadcsv
from libs.matplotlib_figure_style import *

import matplotlib.pyplot as plt

sc.logging.print_header()
print(f"squidpy=={sq.__version__}")

In [ ]:
# load the pre-processed dataset
img = sq.datasets.visium_hne_image()

In [ ]:
root0 = Path('/home/flavio/uv/squidpy/')
root_data = root0 / 'data'
root_figure = root0 / 'figures'
root_result = root0 / 'results'

os.listdir(root_figure)

In [ ]:
filename_vis = root_data / "visium_hne.h5ad"
force=False

if filename_vis.exists() and not force:
    adata = ad.read_h5ad(filename_vis)
else:
    adata = sq.datasets.visium_hne_adata()


### Gene count per cell

In [ ]:
df_de = adata.var
print(df_de.shape)

int(df_de.n_cells_by_counts.max())

In [ ]:
df_de.head(5).T

In [ ]:
df_de[df_de.mt==True].head(5).T

In [ ]:
for i, key in enumerate(adata.obs.keys()):
    print(i, key)

In [ ]:
for i, key in enumerate(adata.obsm.keys()):
    print(i, key)

### let’s visualize cluster annotation in spatial context with squidpy.pl.spatial_scatter().

In [ ]:
plt.rcParams['figure.figsize'] = (10, 7)
# plt.rcParams['figure.dpi'] = 240

ax = sq.pl.spatial_scatter(adata, color="cluster", return_ax=True)

filename = root_figure / "spatial_clusters.png"

if not filename.exists():
    ax.figure.savefig(filename, dpi=300, bbox_inches="tight")
    print(f"Plot saved: {filename}")


### Image feature

Visium datasets contain high-resolution images of the tissue that was used for the gene extraction. Using the function squidpy.im.calculate_image_features() you can calculate image features for each Visium spot and create a obs x features matrix in adata that can then be analyzed together with the obs x gene gene expression matrix.

By extracting image features we are aiming to get both similar and complementary information to the gene expression values. Similar information is for example present in the case of a tissue with two different cell types whose morphology is different. Such cell type information is then contained in both the gene expression values and the tissue image features.

Squidpy contains several feature extractors and a flexible pipeline of calculating features of different scales and sizes. There are several detailed examples of how to use squidpy.im.calculate_image_features(). Extract image features provides a good starting point for learning more.

### extract summary features at different crop sizes and scales

to allow the calculation of multi-scale features and segmentation features. For more information on the summary features, also refer to Extract [summary features](https://squidpy.readthedocs.io/en/latest/notebooks/examples/image/compute_summary_features.html).

In [ ]:
for i, key in enumerate(adata.obsm.keys()):
    print(i, key)

In [ ]:
force=False

if "features" not in adata.obsm.keys() or force:

    # calculate features for different scales (higher value means more context)
    for scale in [1.0, 2.0]:
        feature_name = f"features_summary_scale{scale}"
        print(">>>", feature_name)
        sq.im.calculate_image_features(
            adata,
            img.compute(),
            features="summary",
            key_added=feature_name,
            n_jobs=12,
            scale=scale,
        )


    # combine features in one dataframe
    adata.obsm["features"] = pd.concat(
        [adata.obsm[f] for f in adata.obsm.keys() if "features_summary" in f], axis="columns",
    )
    # make sure that we have no duplicated feature names in the combined table
    adata.obsm["features"].columns = ad.utils.make_index_unique(
        adata.obsm["features"].columns
    )

### extracted image features to compute a new cluster annotation

We can use the extracted image features to compute a new cluster annotation. This could be useful to gain insights in similarities across spots based on image morphology.


#### "features_cluster" are the "leinden" regions, what do they represent?

In that tutorial, `features_cluster` is a clustering of the **Visium spots based purely on the tissue image**, not on gene expression.

What's going on:

- `adata.obsm["features"]` comes from `sq.im.calculate_image_features(...)`. For each spot, Squidpy crops the H&E (or fluorescence) image around the spot's coordinates and computes numeric descriptors of that crop. The columns are prefixed by feature type: `summary_*`, `histogram_*`, `texture_*`, `segmentation_*`.
- `like="summary"` keeps only the summary-statistics features — per channel, things like mean, standard deviation, and quantiles of pixel intensity in the crop. So each spot is represented by a small vector describing how dark/light, how variable, and how colored its local patch of tissue looks.
- `cluster_features` then does the standard scanpy pipeline (scale → PCA → kNN graph → Leiden) on that matrix, treating spots as observations and image statistics as "genes".

So each Leiden label ("0", "1", "2", …) is an arbitrary integer identifying a group of spots whose local histology looks alike in terms of staining intensity and contrast — e.g. densely packed, darkly stained nuclei-rich regions vs. pale, sparse white matter vs. background/edge spots. The labels carry no biological name and no spatial constraint; spots in the same cluster can be scattered across the section.

The point of the tutorial is the comparison with `adata.obs["cluster"]` (the expression-derived, anatomically annotated clusters): plotting both side by side shows how much of the anatomy is recoverable from image appearance alone. You typically see partial correspondence — large-scale structures like cortex vs. white matter vs. hippocampus separate reasonably well, while finer expression-defined subregions collapse or split differently. Adding `texture` or `segmentation` features (nuclei counts, cell density) usually improves that agreement.

In [ ]:
# helper function returning a clustering
def cluster_features(features: pd.DataFrame, like=None) -> pd.Series:
    """
    Calculate leiden clustering of features.
    Specify filter of features using `like`.
    """
    # filter features
    if like:
        features = features.filter(like=like)
    # create temporary adata to calculate the clustering
    adata = ad.AnnData(features)
    # important - feature values are not scaled, so need to scale them before PCA
    sc.pp.scale(adata)
    # calculate leiden clustering
    sc.pp.pca(adata, n_comps=min(10, features.shape[1] - 1))
    sc.pp.neighbors(adata)
    sc.tl.leiden(adata)

    return adata.obs["leiden"]


### calculate feature clusters

In [ ]:
force=False

if "features_cluster" not in adata.obs.keys() or force:
    adata.obs["features_cluster"] = cluster_features(adata.obsm["features"], like="summary")

In [ ]:
for i, key in enumerate(adata.obs.keys()):
    print(i, key)

In [ ]:
dfc = adata.obs['cluster']
print(type(dfc), dfc.shape)
dfc.head(6)

### There are 257 clusters for Cortex 2

In [ ]:
df2 = dfc[dfc=='Cortex_2']
print(df2.shape)
df2.head(3)

In [ ]:
smat = adata.X
print(smat.shape)
type(smat)

In [ ]:
# obs: 'in_tissue', 'array_row', 'array_col', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'n_counts', 'leiden', 'cluster', 'features_cluster'
# var: 'gene_ids', 'feature_types', 'genome', 'mt', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'n_cells', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm'

nrows=4
ncols=2 
fig, axes = plt.subplots(nrows, ncols, figsize=(12, 8), sharex=True, sharey=True)

nrow=0; ncol=1
measures = ["n_genes_by_counts", "log1p_n_genes_by_counts", "total_counts", "log1p_total_counts"]
for nrow in range(1, nrows+1):
    for ncol in range(1, ncols+1):
        ax = axes[nrow-1, ncol-1]

        measure = measures[nrow-1]
        dimensions = (0, 1) if ncol==1 else (2,3)

        print(nrow, ncol, measure, dimensions)
        sc.pl.pca(adata, color=measure, dimensions=dimensions, ax=ax, show=False)

plt.tight_layout()

### compare feature and gene clusters

sq.pl.spatial_scatter(adata, color=["features_cluster", "cluster"])

In [ ]:
axs = sq.pl.spatial_scatter(
    adata,
    color=["features_cluster", "cluster"],
    return_ax=True,
)
axs = [axs] if not isinstance(axs, (list, tuple)) else axs

for ax in axs:
    leg = ax.get_legend()
    if leg is None:
        continue
    handles = leg.legend_handles          # .legendHandles on matplotlib < 3.7
    labels = [t.get_text() for t in leg.get_texts()]
    leg.remove()
    ax.legend(
        handles, labels,
        loc="upper center", bbox_to_anchor=(0.5, -0.05),
        ncol=4, frameon=False, markerscale=2,
    )

Comparing gene and feature clusters, we notice that in some regions, they look very similar, like the cluster Fiber_tract, or clusters around the Hippocampus seems to be roughly recapitulated by the clusters in image feature space. In others, the feature clusters look different, like in the cortex, where the gene clusters show the layered structure of the cortex, and the features clusters rather seem to show different regions of the cortex.

This is only a simple, comparative analysis of the image features, note that you could also use the image features to e.g. compute a common image and gene clustering by computing a shared neighbors graph (for instance on concatenated PCAs on both feature spaces).

In [ ]:
arr_spatial = adata.obsm['spatial']
arr_spatial[:3, :3]

In [ ]:
dff = adata.obsm['features']
print(dff.shape)
dff.iloc[:3, :6].T

### Liana (Saez lab - Heidelberg)

https://github.com/saezlab/liana

### A. Cluster-level (non-spatial) LR inference

In [ ]:
li.mt.rank_aggregate(
    adata,
    groupby="cluster",               # anatomical spot clusters (15 regions), not cell types
    resource_name="mouseconsensus",  # mouse gene symbols
    expr_prop=0.1,
    use_raw=False,                   # .raw holds counts here!
    n_perms=1000, n_jobs=8,
    verbose=True,
)

### source, target, ligand_complex, receptor_complex, ...magnitude_rank, specificity_rank (lower = stronger)

In [ ]:
res = adata.uns["liana_res"]

fname = "liana_cluster_results.tsv"
filename = root_result / fname

if not filename.exists():
    pdwritecsv(res, fname, root_result, verbose=True)


In [ ]:
top_specific = res[(res.specificity_rank <= 0.01)].sort_values("magnitude_rank")

fname = "liana_cluster_top_specific.tsv"
filename = root_result / fname

if not filename.exists():
    pdwritecsv(top_specific, "liana_cluster_top_specific.tsv", root_result, verbose=True)

print(top_specific.shape)
top_specific.head(6).T

### Save complete h5ad

In [ ]:
force=False

if not filename_vis.exists() or force:
    adata.write_h5ad(filename_vis)
    print(f"Data written to {filename_vis}")

### B. Spatially-resolved LR (per spot, no clusters needed)

- Spot centre spacing here is ~137 px (~100 um); bandwidth=200 px ~ 1-2 rings of neighbours.

In [ ]:
li.ut.spatial_neighbors(adata, bandwidth=200, cutoff=0.1, kernel="gaussian", set_diag=True)

filename_lr = root_data / "visium_hne_liana_bivariate.h5ad"

if filename_lr.exists():
    lrdata = ad.read_h5ad(filename_lr)
else:
    lrdata = li.mt.bivariate(
        adata, resource_name="mouseconsensus",
        local_name="cosine",     # local (per-spot) LR co-localisation score -> lrdata.X
        global_name="morans",    # global spatial autocorrelation per LR -> lrdata.var
        n_perms=100, mask_negatives=False, add_categories=True,
        nz_prop=0.2, use_raw=False, verbose=True,
    )
# lrdata: spots x LR-pairs AnnData; .var has morans / morans_pvals; .layers['pvals'], ['cats']
top_global = lrdata.var.sort_values("morans", ascending=False)

In [ ]:
print(top_global.shape)
top_global.head(6).T

### spatial (bivariate) results: the ligand\^receptor name is the INDEX, so keep it

In [ ]:
fname = "liana_spatial_global_morans.tsv"
filename = root_result / fname

df_mor = lrdata.var.sort_values("morans", ascending=False)

if not filename.exists():
    pdwritecsv(df_mor,fname, root_result, index=True, verbose=True)
    print(f"File written: {filename}")

df_mor.head(6).T

### Reading a file back

Tested sizes: the full table is 190,666 rows. The specificity_rank ≤ 0.01 filter keeps 25,341 rows, and the Moran's I table has 454 rows.



In [ ]:
dfqq = pdreadcsv("liana_cluster_top_specific.tsv", root_result)
dfqq.shape

### Dotplot

In [ ]:
p = li.pl.dotplot(adata=adata, colour="magnitude_rank", size="specificity_rank",
                  inverse_colour=True, inverse_size=True,
                  source_labels=["Hippocampus", "Pyramidal_layer", "Pyramidal_layer_dentate_gyrus"],
                  target_labels=["Cortex_1", "Thalamus_1", "Fiber_tract", "Hippocampus", "Striatum"],
                  filter_fun=lambda x: x["specificity_rank"] <= 0.01,
                  top_n=20, orderby="magnitude_rank", orderby_ascending=True, figure_size=(16, 10))


filename = root_figure / "liana_cluster_dotplot.png"

if not filename.exists():
    p.save(filename, dpi=300, verbose=False)
    print(f"Plot saved: {filename}")

In [ ]:
p.show()

### Squidpy and Scanpy plots. 

Get the axes back with return_ax=True, then save the parent figure:

In [ ]:
ax = sq.pl.spatial_scatter(adata, color="cluster", return_ax=True)
plt.show()

In [ ]:
lista = df_mor.head(8).index.to_list()
lista

In [ ]:
# several panels -> a list of axes; all share one figure
ligand_receptor_list = lista

axs = sq.pl.spatial_scatter(lrdata, color=ligand_receptor_list, cmap="coolwarm", return_ax=True)

filename = root_figure / "liana_spatial_LR.png"

if not filename.exists():
    axs[0].figure.savefig(filename, dpi=300, bbox_inches="tight")
    print(f"Plot saved: {filename}")
    
plt.show()

### Plot: two things to watch for:

1. Save before plt.show(). In Jupyter, show() closes the figure, so saving afterwards writes a blank image.
2. bbox_inches="tight" matters. Without it, legends placed outside the axes, like the ones you move below the plot in cell 25, get cut off.


### Save spatial LIANA object

- That way you don't have to rerun li.mt.bivariate. 
- adata.uns["liana_res"] is already saved inside visium_hne_complete.h5ad if you write that file after running LIANA.


In [ ]:
force=False

if not filename_lr.exists() or force:
    lrdata.write_h5ad(filename_lr)
    print(f"Data written to {filename_lr}")

### Save html

jupyter nbconvert --to html notebooks/squid01_analyze_visium_HE_data.ipynb 